# AgroÓrbit — Pipeline de Dados

Esse notebook centraliza o processo de leitura, limpeza e transformação dos dados brutos,
gerando os arquivos que alimentam o dashboard e os notebooks de Machine Learning.

**Fonte principal:** `dados_caatinga_filtrado.json` (INPE BDQueimadas + CHIRPS UCSB + INMET + IBGE 2024 · 130 microrregiões · 1.073 municípios)  
**Saídas geradas:**
- `dados_finais_3anos.csv` — agregado mensal da Caatinga toda (36 linhas)
- `dados_por_uf.csv` — agregado mensal por estado (9 UFs × 36 meses)
- `dados_por_microrregiao.csv` — dados completos por microrregião (4.680 linhas)

```
dados_caatinga_filtrado.json
        │
        ├──► limpeza e validação
        │
        ├──► agregado Caatinga ──► dados_finais_3anos.csv
        ├──► agregado por UF   ──► dados_por_uf.csv
        └──► por microrregião  ──► dados_por_microrregiao.csv
```

## 1. Importações e configurações

In [1]:
import json
import pandas as pd
import numpy as np
import os

CAMINHO_DATA = '../data'
CAMINHO_JSON = f'{CAMINHO_DATA}/dados_caatinga_filtrado.json'
MESES = ['Jan','Fev','Mar','Abr','Mai','Jun','Jul','Ago','Set','Out','Nov','Dez']
ANOS  = [2022, 2023, 2024]

# verificar se o JSON existe
if os.path.exists(CAMINHO_JSON):
    tamanho = os.path.getsize(CAMINHO_JSON) / 1024
    print(f'[OK] {os.path.basename(CAMINHO_JSON)} ({tamanho:.0f} KB)')
else:
    print(f'[FALTANDO] {CAMINHO_JSON}')
    print('Coloque o arquivo dados_caatinga_filtrado.json dentro da pasta data/')

print('\nbibliotecas carregadas!')

[OK] dados_caatinga_filtrado.json (1034 KB)

bibliotecas carregadas!


## 2. Carregando e validando o JSON

Verificação da integridade dos dados antes de processar:
consistência de anos, meses e campos obrigatórios em cada microrregião.

In [2]:
with open(CAMINHO_JSON, 'r', encoding='utf-8') as f:
    dados_raw = json.load(f)

micros = dados_raw['microrregioes']

# validação básica
erros = []
campos_obrigatorios = ['indice', 'nivel', 'focos', 'precip', 'umidade', 'temp']

for micro_id, v in micros.items():
    for ano in ['2022', '2023', '2024', 'media']:
        if ano not in v['meses']:
            erros.append(f'{micro_id}: ano {ano} ausente')
            continue
        for mes in range(1, 13):
            m = v['meses'][ano].get(str(mes), {})
            for campo in campos_obrigatorios:
                if campo not in m:
                    erros.append(f'{micro_id}/{ano}/mes{mes}: campo {campo} ausente')

print('VALIDAÇÃO DO JSON')
print('=' * 45)
print(f'Microrregiões: {len(micros)}')
print(f'Erros encontrados: {len(erros)}')
if erros:
    for e in erros[:10]:
        print(f'  ⚠ {e}')
else:
    print('  Nenhum erro — dados completos!')

# contagem por estado
from collections import Counter
uf_count = Counter(v['uf'] for v in micros.values())
print(f'\nMicrorregiões por estado:')
for uf, n in sorted(uf_count.items()):
    print(f'  {uf}: {n}')

VALIDAÇÃO DO JSON
Microrregiões: 130
Erros encontrados: 0
  Nenhum erro — dados completos!

Microrregiões por estado:
  AL: 7
  BA: 21
  CE: 33
  MG: 3
  PB: 21
  PE: 12
  PI: 8
  RN: 18
  SE: 7


## 3. Pipeline — transformação para DataFrame completo

In [3]:
registros = []

for micro_id, v in micros.items():
    for ano in ANOS:
        for mes in range(1, 13):
            m = v['meses'][str(ano)][str(mes)]
            registros.append({
                'micro_id'    : micro_id,
                'micro_nome'  : v['nome'],
                'uf'          : v['uf'],
                'mesorregiao' : v['mesorregiao'],
                'lat'         : v['lat'],
                'lon'         : v['lon'],
                'semiarido'   : v['semiarido'],
                'pct_caatinga': v.get('pct_caatinga', 100.0),
                'ano'         : ano,
                'mes'         : mes,
                'mes_nome'    : MESES[mes - 1],
                'focos'       : int(m['focos']),
                'precip'      : float(m['precip']),
                'umidade'     : float(m['umidade']),
                'temp'        : float(m['temp']),
                'indice'      : float(m['indice']),
                'nivel'       : m['nivel'],
            })

df_completo = pd.DataFrame(registros)

# nível como categoria ordenada
ordem_nivel = ['BAIXO', 'MEDIO', 'ALTO', 'CRITICO']
df_completo['nivel'] = pd.Categorical(df_completo['nivel'], categories=ordem_nivel, ordered=True)

print(f'DataFrame completo: {df_completo.shape}')
print(f'  130 microrregiões × 12 meses × 3 anos = {130*12*3} registros')
print(f'\nTipos de dados:')
print(df_completo.dtypes)
print(f'\nValores nulos:')
nulos = df_completo.isnull().sum()
print(nulos[nulos > 0] if nulos.sum() > 0 else '  Nenhum valor nulo!')

DataFrame completo: (4680, 17)
  130 microrregiões × 12 meses × 3 anos = 4680 registros

Tipos de dados:
micro_id             str
micro_nome           str
uf                   str
mesorregiao          str
lat              float64
lon              float64
semiarido           bool
pct_caatinga     float64
ano                int64
mes                int64
mes_nome             str
focos              int64
precip           float64
umidade          float64
temp             float64
indice           float64
nivel           category
dtype: object

Valores nulos:
  Nenhum valor nulo!


## 4. Limpeza e tratamento de anomalias

In [4]:
print('LIMPEZA DOS DADOS')
print('=' * 45)

# 1. Verificar valores fora do range esperado
anomalias = 0

# Índice de risco do índice (0 a 1)
fora_indice = df_completo[(df_completo['indice'] < 0) | (df_completo['indice'] > 1)]
print(f'Índice fora de [0,1]: {len(fora_indice)} registros')
anomalias += len(fora_indice)

# Temperatura razoável para a Caatinga (15°C a 45°C)
fora_temp = df_completo[(df_completo['temp'] < 15) | (df_completo['temp'] > 45)]
print(f'Temperatura fora de [15,45]°C: {len(fora_temp)} registros')
anomalias += len(fora_temp)

# Umidade (0 a 100%)
fora_umid = df_completo[(df_completo['umidade'] < 0) | (df_completo['umidade'] > 100)]
print(f'Umidade fora de [0,100]%: {len(fora_umid)} registros')
anomalias += len(fora_umid)

# Focos negativos
focos_neg = df_completo[df_completo['focos'] < 0]
print(f'Focos negativos: {len(focos_neg)} registros')
anomalias += len(focos_neg)

# 2. Nota sobre 2022 sem focos
focos_2022 = df_completo[df_completo['ano'] == 2022]['focos'].sum()
print(f'\nNota: focos 2022 = {focos_2022}')
print('  2022 não tem dados de focos (encoding incompatível no TerraBrasilis)')
print('  Chuva, umidade e temperatura de 2022 estão completos')

print(f'\nTotal de anomalias encontradas: {anomalias}')
print('Dados prontos para uso!' if anomalias == 0 else 'Revisar anomalias antes de continuar')

LIMPEZA DOS DADOS
Índice fora de [0,1]: 0 registros
Temperatura fora de [15,45]°C: 0 registros
Umidade fora de [0,100]%: 0 registros
Focos negativos: 0 registros

Nota: focos 2022 = 269317
  2022 não tem dados de focos (encoding incompatível no TerraBrasilis)
  Chuva, umidade e temperatura de 2022 estão completos

Total de anomalias encontradas: 0
Dados prontos para uso!


## 5. Saída 1 — dados_finais_3anos.csv (agregado Caatinga toda)

In [5]:
# Média mensal de toda a Caatinga — mesmo formato do CSV original
# 36 registros: 12 meses × 3 anos
df_agregado = df_completo.groupby(['ano', 'mes']).agg(
    mes_nome = ('mes_nome', 'first'),
    focos    = ('focos', 'sum'),
    precip   = ('precip', 'mean'),
    umidade  = ('umidade', 'mean'),
    temp     = ('temp', 'mean'),
    indice   = ('indice', 'mean'),
).reset_index()

df_agregado = df_agregado[['ano','mes','mes_nome','focos','precip','umidade','temp','indice']]
df_agregado = df_agregado.sort_values(['ano','mes']).reset_index(drop=True)

# Arredondando para 3 casas decimais
for col in ['precip','umidade','temp','indice']:
    df_agregado[col] = df_agregado[col].round(3)

# Salvando
caminho_saida = f'{CAMINHO_DATA}/dados_finais_3anos.csv'
df_agregado.to_csv(caminho_saida, index=False)

print(f'[OK] dados_finais_3anos.csv salvo')
print(f'     Shape: {df_agregado.shape}')
print(f'\nPrévia (Outubro — mês mais crítico):')
print(df_agregado[df_agregado['mes'] == 10].to_string(index=False))

[OK] dados_finais_3anos.csv salvo
     Shape: (36, 8)

Prévia (Outubro — mês mais crítico):
 ano  mes mes_nome  focos  precip  umidade   temp  indice
2022   10      Out  95845  12.045   66.402 26.403   0.412
2023   10      Out 177219   6.865   64.839 26.623   0.594
2024   10      Out 108577  14.675   67.074 26.645   0.557


## 6. Saída 2 — dados_por_uf.csv (agregado por estado)

In [6]:
# Agregado por UF e mês — útil para análises regionais
df_por_uf = df_completo.groupby(['ano', 'mes', 'uf']).agg(
    mes_nome       = ('mes_nome', 'first'),
    focos          = ('focos', 'sum'),
    precip         = ('precip', 'mean'),
    umidade        = ('umidade', 'mean'),
    temp           = ('temp', 'mean'),
    indice         = ('indice', 'mean'),
    n_microrregioes= ('micro_id', 'count'),
).reset_index()

df_por_uf = df_por_uf.sort_values(['ano','mes','uf']).reset_index(drop=True)

for col in ['precip','umidade','temp','indice']:
    df_por_uf[col] = df_por_uf[col].round(3)

caminho_uf = f'{CAMINHO_DATA}/dados_por_uf.csv'
df_por_uf.to_csv(caminho_uf, index=False)

print(f'[OK] dados_por_uf.csv salvo')
print(f'     Shape: {df_por_uf.shape}')
print(f'\nTop 5 — estado + mês com mais focos (2023 e 2024):')
top = df_por_uf[df_por_uf['ano'] >= 2023].nlargest(5, 'focos')[['ano','mes_nome','uf','focos','indice']]
print(top.to_string(index=False))

[OK] dados_por_uf.csv salvo
     Shape: (324, 10)

Top 5 — estado + mês com mais focos (2023 e 2024):
 ano mes_nome uf  focos  indice
2023      Out BA  69800   0.696
2023      Out PI  57846   0.795
2024      Out BA  45526   0.600
2023      Set BA  44908   0.671
2024      Set PI  37350   0.747


## 7. Saída 3 — dados_por_microrregiao.csv (dados completos)

In [7]:
# Dataset completo com todas as 130 microrregiões
# Usado pelos notebooks de ML (03, 04, 05)
colunas_saida = [
    'ano','mes','mes_nome','micro_id','micro_nome','uf','mesorregiao',
    'lat','lon','semiarido','pct_caatinga',
    'focos','precip','umidade','temp','indice','nivel'
]

df_micro = df_completo[colunas_saida].copy()
df_micro = df_micro.sort_values(['micro_id','ano','mes']).reset_index(drop=True)

# nivel como string para salvar corretamente
df_micro['nivel'] = df_micro['nivel'].astype(str)

caminho_micro = f'{CAMINHO_DATA}/dados_por_microrregiao.csv'
df_micro.to_csv(caminho_micro, index=False)

print(f'[OK] dados_por_microrregiao.csv salvo')
print(f'     Shape: {df_micro.shape}')
print(f'     Colunas: {list(df_micro.columns)}')

[OK] dados_por_microrregiao.csv salvo
     Shape: (4680, 17)
     Colunas: ['ano', 'mes', 'mes_nome', 'micro_id', 'micro_nome', 'uf', 'mesorregiao', 'lat', 'lon', 'semiarido', 'pct_caatinga', 'focos', 'precip', 'umidade', 'temp', 'indice', 'nivel']


## 8. Validação final — checando os 3 arquivos gerados

In [8]:
print('VALIDAÇÃO FINAL')
print('=' * 50)

arquivos_gerados = {
    'dados_finais_3anos.csv'    : (36,  8),
    'dados_por_uf.csv'          : (324, 9),   # 9 UFs × 36 meses
    'dados_por_microrregiao.csv': (4680, 17),  # 130 × 36 meses
}

todos_ok = True
for nome, (linhas_esp, cols_esp) in arquivos_gerados.items():
    caminho = f'{CAMINHO_DATA}/{nome}'
    if os.path.exists(caminho):
        df_check = pd.read_csv(caminho)
        ok = df_check.shape[0] == linhas_esp
        status = 'OK' if ok else f'AVISO — esperado {linhas_esp} linhas'
        print(f'  [{status}] {nome}: {df_check.shape}')
        if not ok: todos_ok = False
    else:
        print(f'  [FALTANDO] {nome}')
        todos_ok = False

print()
if todos_ok:
    print('Pipeline concluído com sucesso!')
    print('Os 3 arquivos estão prontos para os notebooks 03, 04 e 05.')
else:
    print('Revisar erros acima antes de continuar.')

VALIDAÇÃO FINAL
  [OK] dados_finais_3anos.csv: (36, 8)
  [OK] dados_por_uf.csv: (324, 10)
  [OK] dados_por_microrregiao.csv: (4680, 17)

Pipeline concluído com sucesso!
Os 3 arquivos estão prontos para os notebooks 03, 04 e 05.


## 9. Resumo do pipeline

**Arquivos gerados:**

| Arquivo | Registros | Uso |
|---|---|---|
| `dados_finais_3anos.csv` | 36 | Dashboard principal, notebooks 03/04/05 |
| `dados_por_uf.csv` | 324 | Análises regionais por estado |
| `dados_por_microrregiao.csv` | 4.680 | ML detalhado por microrregião |

**Diferenças em relação ao pipeline anterior (Caatinga):**
- Fonte unificada: JSON em vez de 8 CSVs separados
- Escopo ampliado: 130 microrregiões vs. 1 estação (Semiárido brasileiro)
- 9 estados vs. 4 estados do Caatinga
- Mês mais crítico: Outubro (Caatinga) vs. Setembro (Caatinga)

**Próximo passo:** Notebook 03 — Modelo preditivo de risco com Random Forest